# Pikachu robust RL — Colab T4
This notebook pins both repositories, verifies the production engine, resumes only verified episode-boundary checkpoints from Drive, and keeps validation separate from any sealed final set.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_URL = 'https://github.com/jimin326/skku_pikachu.git'
PROJECT_REF = 'robust-rl-colab'  # pinned branch containing the RL system
GAME_URL = 'https://github.com/SKKU-x-HYU-SW-Competition/leonyi-volleyball.git'
GAME_COMMIT = '1f3cecb90aca174ffc42ac6be4c384cc725d9e91'
RECOVERY = '/content/drive/MyDrive/pikachu_rl/checkpoints'


In [ ]:
!if [ ! -d /content/skku_pikachu/.git ]; then git clone {PROJECT_URL} /content/skku_pikachu; fi
!git -C /content/skku_pikachu fetch origin {PROJECT_REF}
!git -C /content/skku_pikachu checkout {PROJECT_REF}
!if [ ! -d /content/leonyi-volleyball/.git ]; then git clone {GAME_URL} /content/leonyi-volleyball; fi
%cd /content/leonyi-volleyball
!git checkout {GAME_COMMIT}
%cd /content/skku_pikachu
!python -m pip install -q -r requirements-rl.txt
!node scripts/setup_rl_engine.mjs /content/leonyi-volleyball


In [ ]:
!node bot-dev/rl/physics_clamp_smoke.mjs
!node bot-dev/rl/production_differential.mjs --game-root /content/leonyi-volleyball
!node bot-dev/rl/env_smoke.mjs
!python bot-dev/rl/bridge_smoke.py
!python bot-dev/rl/ppo_tests.py
!python bot-dev/rl/eval/test_schema.py
!python bot-dev/rl/eval/test_stats.py
!node bot-dev/rl/eval/paired_eval_smoke.mjs
import torch
assert torch.cuda.is_available(), 'Select a T4 GPU runtime before training'
print(torch.cuda.get_device_name(0))


In [ ]:
# Optional v4 behavior-cloning initialization. Increase decisions for a real run.
RUN_BC = False
if RUN_BC:
    !node bot-dev/rl/collect_bc.mjs --decisions=500000 --output=/content/drive/MyDrive/pikachu_rl/bc/v4.jsonl
    !python bot-dev/rl/bc_pretrain.py /content/drive/MyDrive/pikachu_rl/bc/v4.jsonl /content/drive/MyDrive/pikachu_rl/bc/v4_ff.pt --epochs=10 --device=cuda


In [ ]:
import hashlib, json, os
latest = os.path.join(RECOVERY, 'latest.json')
resume_args = []
if os.path.exists(latest):
    pointer = json.load(open(latest))
    checkpoint = os.path.join(RECOVERY, pointer['checkpoint'])
    assert hashlib.sha256(open(checkpoint, 'rb').read()).hexdigest() == pointer['sha256']
    resume_args = ['--resume', checkpoint]
elif RUN_BC:
    resume_args = ['--initial-model', '/content/drive/MyDrive/pikachu_rl/bc/v4_ff.pt']
resume_args


In [ ]:
# Benchmark 8/16/32 total envs first; Node physics/IPC, not VRAM, is usually the bottleneck.
TOTAL_STEPS = 2_000_000
args = ['python','bot-dev/rl/ppo_train.py','--device','auto','--workers','4','--envs-per-worker','4',
        '--total-steps',str(TOTAL_STEPS),'--checkpoint-dir','/content/checkpoints',
        '--recovery-dir',RECOVERY,'--save-every-minutes','30'] + resume_args
import subprocess
subprocess.run(args, check=True)


In [ ]:
# Validation only. Do not put sealed-final seeds in this notebook.
pointer = json.load(open(os.path.join(RECOVERY, 'latest.json')))
checkpoint = os.path.join(RECOVERY, pointer['checkpoint'])
!mkdir -p /content/export src/code-here
!python bot-dev/rl/export_policy.py {checkpoint} /content/export/Robust_RL_v1.js
!python bot-dev/rl/export_policy_test.py {checkpoint} /content/export/Robust_RL_v1.js
!node bot-dev/rl/export_env_smoke.mjs /content/export/Robust_RL_v1.js
!node bot-dev/rl/eval/paired_eval.mjs --candidate=/content/export/Robust_RL_v1.js --output=/content/export/validation.jsonl
!python bot-dev/rl/eval/stats.py /content/export/validation.jsonl --output=/content/export/validation_stats.json
!node --expose-gc bot-dev/rl/eval/runtime_bench.mjs --candidate=/content/export/Robust_RL_v1.js > /content/export/runtime.json
print('Copy to src/code-here only after the pre-registered acceptance gate passes.')
